In [7]:
import requests
import pandas as pd
import datetime
import time

print("1. Building taxonomy map...")
subfields_url = "https://api.openalex.org/subfields?per_page=200"
all_subfields = []
page = 1

while True:
    try:
        r = requests.get(f"{subfields_url}&page={page}")
        if r.status_code != 200: break
        data = r.json().get('results', [])
        if not data: break
        all_subfields.extend(data)
        page += 1
    except:
        break

hierarchy_map = {}
for sf in all_subfields:
    sf_id = sf['id'].split('/')[-1] # e.g., "1100"
    field = sf.get('field', {})
    domain = sf.get('domain', {}) 
    if not domain: domain = field.get('domain', {})

    hierarchy_map[sf_id] = {
        'Subfield': sf['display_name'],
        'Field': field.get('display_name', 'Unknown'),
        'Domain': domain.get('display_name', 'Unknown')
    }
print(f"   Taxonomy built ({len(hierarchy_map)} subfields).")

# --- STEP 2: Define Universities ---
universities = {
    "U_Michigan": "I27837315",
    "Harvard": "I136199984",
    "Yale": "I32971472",
    "Stanford": "I97018004",
    "Cornell": "I205783295",
    "Georgetown": "I184565670",
    "Princeton": "I20089843",
    "Johns_Hopkins": "I145311948",
    "Carnegie_Mellon": "I74973139",
    "NYU": "I57206974",
    "UCLA": "I161318765",
    "UWashington": "I201448701",
    "UW_Madison": "I135310074",
    "U_Utah": "I223532165",
    "Penn_State": "I130769515",
    "UVA": "I51556381",
    "UMD": "I66946132",
    "V_Tech": "I859038795",
    "Vanderbilt": "I200719446",
    "U_Florida": "I33213144"
}

for label, uni_id in universities.items():
    try:
        r = requests.get(f"https://api.openalex.org/institutions/{uni_id}")
        if r.status_code == 200:
            actual_name = r.json().get('display_name', 'Unknown')
            # Check if your label vaguely matches the actual name
            status = "✅" if label.lower().replace("_", " ") in actual_name.lower() or \
                             actual_name.split(' ')[0].lower() in label.lower() else "❌ MISMATCH"
            print(f"{label:<20} | {uni_id:<12} | {status} {actual_name}")
        else:
            print(f"{label:<20} | {uni_id:<12} | ⚠️  Invalid ID")
    except Exception as e:
        print(f"{label:<20} | Error")
    
    time.sleep(0.2)

1. Building taxonomy map...
   Taxonomy built (252 subfields).
U_Michigan           | I27837315    | ❌ MISMATCH University of Michigan
Harvard              | I136199984   | ✅ Harvard University
Yale                 | I32971472    | ✅ Yale University
Stanford             | I97018004    | ✅ Stanford University
Cornell              | I205783295   | ✅ Cornell University
Georgetown           | I184565670   | ✅ Georgetown University
Princeton            | I20089843    | ✅ Princeton University
Johns_Hopkins        | I145311948   | ✅ Johns Hopkins University
Carnegie_Mellon      | I74973139    | ✅ Carnegie Mellon University
NYU                  | I57206974    | ❌ MISMATCH New York University
UCLA                 | I161318765   | ❌ MISMATCH University of California, Los Angeles
UWashington          | I201448701   | ❌ MISMATCH University of Washington
UW_Madison           | I135310074   | ❌ MISMATCH University of Wisconsin–Madison
U_Utah               | I223532165   | ❌ MISMATCH University of Ut

In [6]:
import requests
import urllib.parse

# 1. Define the name you are looking for
target_name = "University of Florida"

# 2. Search OpenAlex
query = urllib.parse.quote(target_name)
url = f"https://api.openalex.org/institutions?search={query}"
r = requests.get(url)

# 3. Print the result
if r.status_code == 200:
    results = r.json().get('results', [])
    if results:
        top_hit = results[0]
        print(f"Name: {top_hit['display_name']}")
        print(f"ID:   {top_hit['id'].split('/')[-1]}")
    else:
        print("No results found.")

Name: University of Florida
ID:   I33213144


In [8]:
# --- STEP 3: Fetch Data Instantly ---
current_year = datetime.date.today().year
start_year = current_year - 10
date_filter = f"from_publication_date:{start_year}-01-01,to_publication_date:{current_year}-12-31"

print(f"\n2. Fetching counts for {date_filter}...")

for uni_name, uni_id in universities.items():
    print(f"   Processing: {uni_name}...", end=" ")

    
    url = (
        f"https://api.openalex.org/works?"
        f"filter=authorships.institutions.lineage:{uni_id},{date_filter}"
        f"&group_by=primary_topic.subfield.id"
    )
    
    try:
        r = requests.get(url)
        if r.status_code == 200:
            data = r.json()
            groups = data.get('group_by', [])
            
            rows = []
            total_papers = 0
            
            for group in groups:
                # The 'key' is the URL, e.g., "https://openalex.org/subfields/1100"
                # We just need the "1100" part
                sf_id = group['key'].split('/')[-1]
                count = group['count']
                total_papers += count
                
                info = hierarchy_map.get(sf_id)
                if info:
                    rows.append({
                        'Domain': info['Domain'],
                        'Group': info['Field'],
                        'Field': info['Subfield'],
                        'numpub': count
                    })
            
            if rows:
                df = pd.DataFrame(rows)
                df = df.sort_values(by=['Domain', 'Group', 'Field'])
                
                # Save the file
                df.to_csv(f"{uni_name}.csv", index=False)
                print(f"Done! ({total_papers} papers)")
            else:
                print("No data found.")
        else:
            print(f"Error {r.status_code}")
            
    except Exception as e:
        print(f"Failed: {e}")
        
    # Be polite to the API
    time.sleep(0.5)

print("\nAll files generated.")


2. Fetching counts for from_publication_date:2016-01-01,to_publication_date:2026-12-31...
   Processing: U_Michigan... Done! (220460 papers)
   Processing: Harvard... Done! (366788 papers)
   Processing: Yale... Done! (139912 papers)
   Processing: Stanford... Done! (211101 papers)
   Processing: Cornell... Done! (155888 papers)
   Processing: Georgetown... Done! (43995 papers)
   Processing: Princeton... Done! (64875 papers)
   Processing: Johns_Hopkins... Done! (194738 papers)
   Processing: Carnegie_Mellon... Done! (50828 papers)
   Processing: NYU... Done! (108922 papers)
   Processing: UCLA... Done! (135218 papers)
   Processing: UWashington... Done! (179914 papers)
   Processing: UW_Madison... Done! (113179 papers)
   Processing: U_Utah... Done! (86732 papers)
   Processing: Penn_State... Done! (108938 papers)
   Processing: UVA... Done! (66964 papers)
   Processing: UMD... Done! (79262 papers)
   Processing: V_Tech... Done! (61164 papers)
   Processing: Vanderbilt... Done! (594